In [0]:
# COMMAND ----------

from pyspark.sql import functions as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

CATALOG = "aml_engine"
SCHEMA = "aml_poc"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_alerts"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_alerts"

S3_BASE_PATH = "s3://zubair-s3-demo/raw_dataset/aml"
S3_DELTA_PATH = f"{S3_BASE_PATH}/delta_tables"

SILVER_ALERTS_PATH = (
    f"{S3_DELTA_PATH}/silver_alerts"
)


# ============================================================
# 2. READ BRONZE ALERTS
# ============================================================

bronze_alerts_df = spark.table(BRONZE_TABLE)

print(f"Bronze table : {BRONZE_TABLE}")
print(f"Record count : {bronze_alerts_df.count()}")

display(bronze_alerts_df.limit(10))

In [0]:
# COMMAND ----------

silver_alerts_df = (
    bronze_alerts_df

    .select(
        F.col("ALERT_ID")
            .cast("long")
            .alias("alert_id"),

        F.lower(
            F.trim(F.col("ALERT_TYPE"))
        ).alias("alert_type"),

        F.col("IS_FRAUD")
            .cast("boolean")
            .alias("is_fraud"),

        F.col("TX_ID")
            .cast("long")
            .alias("tx_id"),

        F.col("SENDER_ACCOUNT_ID")
            .cast("long")
            .alias("sender_account_id"),

        F.col("RECEIVER_ACCOUNT_ID")
            .cast("long")
            .alias("receiver_account_id"),

        F.upper(
            F.trim(F.col("TX_TYPE"))
        ).alias("tx_type"),

        F.col("TX_AMOUNT")
            .cast("double")
            .alias("tx_amount"),

        # Preserve the dataset's time-step
        F.col("TIMESTAMP")
            .cast("long")
            .alias("event_time")
    )
)

display(silver_alerts_df.limit(10))

In [0]:
# COMMAND ----------

silver_alerts_df = (
    silver_alerts_df

    # Alert ID is mandatory
    .filter(
        F.col("alert_id").isNotNull()
    )

    # Alert must reference a transaction
    .filter(
        F.col("tx_id").isNotNull()
    )

    # Alert type is required
    .filter(
        F.col("alert_type").isNotNull()
    )

    # Account IDs should exist
    .filter(
        F.col("sender_account_id").isNotNull()
    )

    .filter(
        F.col("receiver_account_id").isNotNull()
    )

    # Transaction amount must be valid
    .filter(
        F.col("tx_amount").isNotNull()
    )

    .filter(
        F.col("tx_amount") > 0
    )
)

In [0]:
# COMMAND ----------

silver_alerts_df = (
    silver_alerts_df
    .dropDuplicates(["alert_id", "tx_id"])
)

In [0]:
# COMMAND ----------

silver_alerts_df = (
    silver_alerts_df
    .withColumn(
        "_silver_processed_timestamp",
        F.current_timestamp()
    )
)

In [0]:
# COMMAND ----------

(
    silver_alerts_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "path",
        SILVER_ALERTS_PATH
    )
    .saveAsTable(SILVER_TABLE)
)

print("Silver Alerts table created successfully.")
print(f"Unity Catalog table : {SILVER_TABLE}")
print(f"S3 location         : {SILVER_ALERTS_PATH}")